Підготовка

In [1]:
import os, re, glob, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

os.makedirs("results/figures", exist_ok=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

N = len(df)
TRAIN_W, VAL_W, TEST_W = (0, 17), (18, 21), (22, 26)
inw = lambda r: ((df.week >= r[0]) & (df.week <= r[1])).values
m_tr, m_va, m_te = inw(TRAIN_W), inw(VAL_W), inw(TEST_W)
assert (m_tr | m_va | m_te).all() and not (m_tr & m_va).any() and not (m_va & m_te).any()
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)

ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts))
rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
y = df.isFraud.values
assert y.sum() == 20663

X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
spw = (y[m_tr] == 0).sum() / (y[m_tr] == 1).sum()

def metrics(y_true, p):
    k = int(y_true.sum()); o = np.argsort(-p)
    top1 = o[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p),
            "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[o[:k]].sum() / k,
            "precision@1%": y_true[top1].mean(),
            "base_rate": y_true.mean()}

major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")

def run_xgb_X(Xf, seed):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                      eval_metric="aucpr", early_stopping_rounds=50,
                      random_state=seed, **DEV)
    m.fit(Xf[m_tr], y[m_tr], eval_set=[(Xf[m_va], y[m_va])], verbose=False)
    r = metrics(y[m_te], m.predict_proba(Xf[m_te])[:, 1])
    r.update(seed=seed, best_iter=m.best_iteration, val_aucpr=m.best_score)
    return r, m

print("готово:", X.shape, "| xgboost", xgb.__version__)

готово: (590540, 434) | xgboost 3.4.1


Абляція вікна

In [6]:
def label_feats_w(cols, W_days, DL=7):
    valid = df[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    nk = np.full(N, np.nan, "float32"); fk = np.full(N, np.nan, "float32")
    for pos in df.loc[valid, cols].groupby(cols, sort=False).indices.values():
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, ff = ts[rows], y[rows]; cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")
        hi = np.minimum(np.searchsorted(tt, tt - DL * 86400, side="right"), a)
        lo = np.searchsorted(tt, tt - W_days * 86400, side="left") if W_days else np.zeros_like(hi)
        lo = np.minimum(lo, hi)
        nk[rows] = hi - lo; fk[rows] = cs[hi] - cs[lo]
    return nk, fk

KEYS = {"c1": ["card1"], "u2": ["card1", "addr1"], "u3": ["card1", "addr1", "P_emaildomain"]}
res = []
for W in (30, 90, 0):                               # 0 = необмежена історія
    t0 = time.time(); LFw = {}
    for rel, cols in KEYS.items():
        nk, fk = label_feats_w(cols, W)
        LFw[f"{rel}_nk"], LFw[f"{rel}_fk"] = nk, fk
        LFw[f"{rel}_rate"] = fk / np.where(nk > 0, nk, np.nan)
    Xf = pd.concat([X, pd.DataFrame(LFw)], axis=1)
    for sd in range(5):
        r, _ = run_xgb_X(Xf, sd); r["вікно, днів"] = W if W else "необмежене"; res.append(r)
    print("вікно", W or "необмежене", "готово,", round(time.time() - t0), "с", flush=True)
res = pd.DataFrame(res)
tab = res.groupby("вікно, днів")[["pr_auc", "roc_auc", "recall@k", "precision@1%"]].agg(["mean", "std"]).round(3)
res.to_csv("results/ieee_cis_label_window.csv", index=False); display(tab)

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [16:35:57] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


вікно 30 готово, 145 с
вікно 90 готово, 147 с
вікно необмежене готово, 148 с


pr_auc        roc_auc        recall@k        precision@1%       
              mean    std    mean    std     mean    std         mean    std
вікно, днів                                                                 
30           0.595  0.005   0.927  0.003    0.561  0.006        0.903  0.006
90           0.590  0.001   0.924  0.003    0.560  0.003        0.904  0.004
необмежене   0.572  0.005   0.915  0.005    0.553  0.006        0.894  0.006

In [7]:
res.groupby("вікно, днів")["val_aucpr"].agg(["mean", "std"]).round(4)

,mean,std
"вікно, днів",,
30,0.6397,0.0050
90,0.6418,0.0051
необмежене,0.6224,0.0038
